# Ropedia Academy — A2 · 2D & 3D pose estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A2.ipynb)

> **Covers the 2D→3D pipeline: heatmap soft-argmax for keypoints, then a lifting network to 3D, with the depth-ambiguity point made explicit.**
>
> 覆盖 2D→3D 全流程：用热图 soft-argmax 取关键点，再用 lifting 网络抬升到 3D，并点明深度歧义。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A2

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# 2D pose: a per-keypoint HEATMAP -> sub-pixel coordinate (soft-argmax, differentiable).
H = W = 16
def soft_argmax(hm):                           # hm: (K,H,W) score maps
    K = hm.shape[0]
    p = F.softmax(hm.reshape(K, -1), -1).reshape(K, H, W)   # spatial probability
    xs = torch.arange(W).float(); ys = torch.arange(H).float()
    x = (p.sum(1) * xs).sum(-1)                 # expected column
    y = (p.sum(2) * ys).sum(-1)                 # expected row
    return torch.stack([x, y], -1)             # (K,2)

heatmaps = torch.randn(17, H, W)               # 17 body keypoints
kp2d = soft_argmax(heatmaps)
print("2D keypoints from heatmaps:", tuple(kp2d.shape))

# 3D is ill-posed from one image (depth ambiguity) -> LIFT 2D to 3D with a body prior.
lifter = nn.Sequential(nn.Linear(17*2, 128), nn.ReLU(), nn.Linear(128, 17*3))
kp3d = lifter(kp2d.flatten()).reshape(17, 3)
print("lifted 3D joints:", tuple(kp3d.shape))
print("heatmaps keep uncertainty/structure; lifting + (video over time) fix depth")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A2
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks